# TP3 — Preprocesamiento de Tweets

**Alumno:** Gonzalo Zarazaga

---

Limpieza y tokenización específica para tweets, aplicada de forma idéntica a los dos archivos del corpus, según lo decidido en `00_lectura_y_discovery.ipynb`:

- `training.1600000.processed.noemoticon.csv` completo (obligatorio) → deduplicado por texto exacto.
- `testdata.manual.2009.06.14.csv` filtrado (`polarity != 2`, se descartan los neutros) → test externo, out-of-sample.

Este notebook **no ajusta ningún parámetro estadístico** (no vectoriza, no entrena) — es limpieza determinística basada en reglas, por lo que aplicarla igual a ambos archivos no viola la disciplina fit/transform documentada en el README.

**Insumo:** `data/raw/training.1600000.processed.noemoticon.csv`, `data/raw/testdata.manual.2009.06.14.csv`
**Salida:** `data/processed/train_processed.csv`, `data/processed/test_manual_processed.csv`

## 1. Carga de datos

In [1]:
import re
import html
import pandas as pd
from nltk.tokenize import TweetTokenizer

COLS = ["polarity", "id", "date", "query", "user", "text"]

df_train = pd.read_csv("../data/raw/training.1600000.processed.noemoticon.csv", encoding="latin-1", names=COLS)
df_test_manual = pd.read_csv("../data/raw/testdata.manual.2009.06.14.csv", encoding="latin-1", names=COLS)

print(f"train:        {df_train.shape[0]:,} filas")
print(f"test manual:  {df_test_manual.shape[0]:,} filas")

train:        1,600,000 filas
test manual:  498 filas


## 2. Filtrado y deduplicación

- Se descartan del archivo manual los registros `polarity == 2` (neutros) — decisión B de `00_lectura_y_discovery.ipynb`.
- Se deduplica el archivo de entrenamiento por texto exacto — evita que retweets/spam sobre-representen frases en el vocabulario.

In [2]:
test_manual_bin = df_test_manual[df_test_manual["polarity"] != 2].copy()
print(f"test manual sin neutros: {len(test_manual_bin)} registros "
      f"(se descartaron {len(df_test_manual) - len(test_manual_bin)} neutros)")

n_antes = len(df_train)
df_train = df_train.drop_duplicates(subset="text", keep="first").copy()
print(f"train deduplicado: {len(df_train):,} registros "
      f"(se descartaron {n_antes - len(df_train):,} duplicados exactos)")

test manual sin neutros: 359 registros (se descartaron 139 neutros)


train deduplicado: 1,581,466 registros (se descartaron 18,534 duplicados exactos)


## 3. Pipeline de limpieza específico para tweets

Cada paso corresponde a una decisión documentada en el README (sección "Pipeline detallado"):

1. **Decodificar entidades HTML** (`&quot;`, `&amp;`, ...) — el dump original quedó con HTML sin escapar (~5,9% de los tweets).
2. **Eliminar URLs** — no aportan señal semántica.
3. **Eliminar emoticones residuales** (`:)`, `:'(`, etc.) — pese al nombre `noemoticon` del archivo, ~1,1% de los tweets todavía conserva el emoticon que originalmente definió la etiqueta. Si queda en el texto, el modelo memoriza el heurístico de etiquetado en vez de aprender sentimiento real.
4. **Tokenizar con `TweetTokenizer`** (NLTK), con `strip_handles=True` (elimina `@usuario`), `reduce_len=True` (normaliza caracteres repetidos: `loooove` → `looove`) y `preserve_case=False` (minúsculas).
5. **Quitar el `#` de los hashtags, conservando la palabra** — `#fail` sí lleva carga semántica, perderla sería tirar señal útil.
6. **Descartar tokens de puntuación pura** (`...`, `!!!`, `-`) que no aportan contenido léxico.

**Nota sobre stopwords:** la remoción de stopwords **no se aplica en este notebook**. Word2Vec se beneficia del contexto natural de la oración (se entrena sobre estos mismos tokens sin filtrar). La remoción de stopwords es específica de los modelos de representación dispersa (BoW/TF-IDF) y se hace en `02_modelos_clasicos.ipynb`, excluyendo explícitamente las negaciones (`not`, `no`, `n't`, `never`) de la lista, ya que invierten la polaridad del tweet.

In [3]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")

EMOTICON_RE = re.compile(r"""
    (?:[<>]?[:;=8][\-o\*']?[\)\]\(\[dDpP/:\}\{@\|\\])
    |
    (?:[\)\]\(\[dDpP/:\}\{@\|\\][\-o\*']?[:;=8][<>]?)
""", re.VERBOSE)

PUNCT_ONLY_RE = re.compile(r"^\W+$")

tknzr = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)


def clean_and_tokenize(text):
    text = html.unescape(text)
    text = URL_RE.sub("", text)
    text = EMOTICON_RE.sub("", text)

    tokens = tknzr.tokenize(text)

    out = []
    for tok in tokens:
        if tok.startswith("#") and len(tok) > 1:
            tok = tok[1:]
        if PUNCT_ONLY_RE.match(tok):
            continue
        out.append(tok)
    return out

## 4. Aplicación del pipeline

Se aplica la misma función a ambos datasets.

In [4]:
df_train["tokens"] = df_train["text"].apply(clean_and_tokenize)
test_manual_bin["tokens"] = test_manual_bin["text"].apply(clean_and_tokenize)
print("Listo.")

Listo.


In [5]:
ejemplos = df_train[["text", "tokens"]].sample(6, random_state=42)
for _, fila in ejemplos.iterrows():
    print(f"RAW:   {fila['text']}")
    print(f"LIMPIO:{fila['tokens']}")
    print()

RAW:   @hlooman Hans I'm an open book you can ask me anything you want, hit up the blog and ask away, I can give you more in depth answer there 
LIMPIO:['hans', "i'm", 'an', 'open', 'book', 'you', 'can', 'ask', 'me', 'anything', 'you', 'want', 'hit', 'up', 'the', 'blog', 'and', 'ask', 'away', 'i', 'can', 'give', 'you', 'more', 'in', 'depth', 'answer', 'there']

RAW:   wuaaah we have strawberries at home 
LIMPIO:['wuaaah', 'we', 'have', 'strawberries', 'at', 'home']

RAW:   @JamesSmithComic Very cool.  And to quote TBS... &quot;Now that's funny&quot;.  
LIMPIO:['very', 'cool', 'and', 'to', 'quote', 'tbs', 'now', "that's", 'funny']

RAW:   #iremember the last sunday 
LIMPIO:['iremember', 'the', 'last', 'sunday']

RAW:   @microilist oh thanks, thats good to know 
LIMPIO:['oh', 'thanks', 'thats', 'good', 'to', 'know']

RAW:   @moppet10 Hi Kate  How are you?
LIMPIO:['hi', 'kate', 'how', 'are', 'you']



## 5. Tweets vacíos tras la limpieza

Algunos tweets consistían únicamente en una URL, una mención o un emoticon — tras la limpieza quedan sin tokens y se descartan.

In [6]:
for nombre, df in [("train", df_train), ("test manual", test_manual_bin)]:
    vacios = (df["tokens"].apply(len) == 0).sum()
    print(f"{nombre}: {vacios} tweets vacíos tras la limpieza ({vacios / len(df) * 100:.2f}%)")

df_train = df_train[df_train["tokens"].apply(len) > 0].copy()
test_manual_bin = test_manual_bin[test_manual_bin["tokens"].apply(len) > 0].copy()

print(f"\ntrain final:       {len(df_train):,} registros")
print(f"test manual final: {len(test_manual_bin):,} registros")

train: 3017 tweets vacíos tras la limpieza (0.19%)
test manual: 0 tweets vacíos tras la limpieza (0.00%)



train final:       1,578,449 registros
test manual final: 359 registros


## 6. Exportar a `data/processed/`

Los tokens se guardan unidos por espacio (ningún token de `TweetTokenizer` contiene espacios internos, por lo que el join es reversible con `.split(" ")`).

In [7]:
df_train_out = df_train[["id", "polarity", "text"]].copy()
df_train_out["tokens"] = df_train["tokens"].apply(lambda toks: " ".join(toks))

test_manual_out = test_manual_bin[["id", "polarity", "text"]].copy()
test_manual_out["tokens"] = test_manual_bin["tokens"].apply(lambda toks: " ".join(toks))

df_train_out.to_csv("../data/processed/train_processed.csv", index=False)
test_manual_out.to_csv("../data/processed/test_manual_processed.csv", index=False)

print(f"Guardado data/processed/train_processed.csv        ({len(df_train_out):,} filas)")
print(f"Guardado data/processed/test_manual_processed.csv   ({len(test_manual_out):,} filas)")

Guardado data/processed/train_processed.csv        (1,578,449 filas)
Guardado data/processed/test_manual_processed.csv   (359 filas)


## 7. Resumen de decisiones

| Decisión | Detalle |
|---|---|
| Deduplicación | Se descartaron duplicados exactos de texto en el archivo de entrenamiento antes de tokenizar |
| Filtrado de neutros | Se descartan `polarity == 2` del archivo manual (decisión B, ver `00_lectura_y_discovery.ipynb`) |
| Limpieza de URLs, HTML, emoticones residuales, menciones | Se eliminan por no aportar señal semántica o por filtrar el heurístico de etiquetado |
| Hashtags | Se conserva la palabra, se descarta el símbolo `#` |
| Stopwords | **No se remueven acá** — se remueven solo para BoW/TF-IDF en `02_modelos_clasicos.ipynb`, excluyendo negaciones |
| Tokenización | `TweetTokenizer` de NLTK (`reduce_len=True`, `strip_handles=True`, `preserve_case=False`) en vez de `word_tokenize` genérico, por su mejor manejo de texto informal de Twitter |
| Salida | `data/processed/train_processed.csv` y `data/processed/test_manual_processed.csv`, con tokens ya limpios unidos por espacio |